In [1]:
import json
import pathlib

import pandas as pd

In [2]:

project_root = pathlib.Path('../../..')
project_root.resolve()

PosixPath('/home/jnban/projects/roanoke-transit')

In [3]:
metroflex_data = project_root / 'data/metroflex'
metroflex_data.resolve()

PosixPath('/home/jnban/projects/roanoke-transit/data/metroflex')

In [4]:
with open(metroflex_data / 'metroflex-addresses.json', 'r') as f:
    address_locations = json.load(f)
address_locations

{'1505 Queen Ann Dr Se\nRoanoke, VA 24014': {'input': {'address_components': {'number': '1505',
    'street': 'Queen Ann',
    'suffix': 'Dr',
    'postdirectional': 'SE',
    'formatted_street': 'Queen Ann Dr SE',
    'city': 'Roanoke',
    'state': 'VA',
    'zip': '24014',
    'country': 'US'},
   'formatted_address': '1505 Queen Ann Dr SE, Roanoke, VA 24014'},
  'results': [{'address_components': {'number': '1505',
     'street': 'Queen Ann',
     'suffix': 'Dr',
     'postdirectional': 'SE',
     'formatted_street': 'Queen Ann Dr SE',
     'city': 'Roanoke',
     'county': 'Roanoke City',
     'state': 'VA',
     'zip': '24014',
     'country': 'US'},
    'address_lines': ['1505 Queen Ann Dr SE', '', 'Roanoke, VA 24014'],
    'formatted_address': '1505 Queen Ann Dr SE, Roanoke, VA 24014',
    'location': {'lat': 37.254114, 'lng': -79.914682},
    'accuracy': 1,
    'accuracy_type': 'rooftop',
    'source': 'Virginia Geographic Information Network (VGIN)'}]},
 '3941 Thirlane Rd\nRo

In [8]:
def lookup(row, **kwargs):
    details = row['Pickup Address Details']
    
    key = row['Pickup Address']
    if key not in address_locations:
        return None
    
    address = address_locations[key]
    
    matches = address['results']
    
    matches = [
        match
        for match in matches
        if match['address_components']['state'] == 'VA'
    ]
        
    if len(matches) == 0:
        raise ValueError(f'Address not found: {details} at {key}')
    if len(matches) == 1:
        return matches[0]
    raise ValueError(f'Multiple address locations found:{details} at {key} {matches}')

df = pd.read_csv(metroflex_data / 'metroflex-2025-02-trip-report-cleaned.csv')
for index, row in df.iterrows():
    match = lookup(row)
    df.loc[index, 'Pickup Lat'] = match['location']['lat']
    df.loc[index, 'Pickup Lng'] = match['location']['lng']
    df.loc[index, 'Pickup addy'] = match['formatted_address'] 
df[df['Pickup LonLat'].isnull()]

ValueError: Multiple address locations found:BJ'S at 1419 Hershberger Rd
Roanoke, VA 24012 [{'address_components': {'number': '1419', 'street': 'Hershberger', 'suffix': 'Rd', 'postdirectional': 'NW', 'formatted_street': 'Hershberger Rd NW', 'city': 'Roanoke', 'county': 'Roanoke City', 'state': 'VA', 'zip': '24012', 'country': 'US'}, 'address_lines': ['1419 Hershberger Rd NW', '', 'Roanoke, VA 24012'], 'formatted_address': '1419 Hershberger Rd NW, Roanoke, VA 24012', 'location': {'lat': 37.316636, 'lng': -79.959878}, 'accuracy': 1, 'accuracy_type': 'rooftop', 'source': 'Virginia Geographic Information Network (VGIN)'}, {'address_components': {'number': '1419', 'street': 'Hershberger', 'suffix': 'Rd', 'postdirectional': 'NW', 'formatted_street': 'Hershberger Rd NW', 'city': 'Roanoke', 'county': 'Roanoke City', 'state': 'VA', 'zip': '24012', 'country': 'US'}, 'address_lines': ['1419 Hershberger Rd NW', '', 'Roanoke, VA 24012'], 'formatted_address': '1419 Hershberger Rd NW, Roanoke, VA 240